# **Training Titan MAC on Stories**
As we saw the result was not good enough for tiny data so lets do it for a real data "roneneldan/TinyStories"

In [ ]:
!pip install datasets transformers

In [ ]:
from datasets import load_dataset
from transformers import GPT2Tokenizer

In [ ]:
dataset = load_dataset("roneneldan/TinyStories", split="train[:1000]") # First 1000 stories

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [ ]:
import torch
import torch.nn as nn

In [ ]:
vocab_size = tokenizer.vocab_size
# print(vocab_size)
embed_dim = 512

In [ ]:
embed = nn.Embedding(vocab_size, embed_dim)
# print(embed)
embed.weight.requires_grad = True

In [ ]:
# LMM & Attention
class LMM(nn.Module):
    def __init__(self, dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, dim)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:

class Attention(nn.Module):
    def __init__(self, dim=512):
        super().__init__()
        self.dim = dim
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)

    def forward(self, x):
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.dim ** 0.5)
        attn_weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, v)
        return output, attn_weights

In [ ]:
class MAC_Layer(nn.Module):
    def __init__(self, dim=512, vocab_size=None):
        super().__init__()
        self.lmm = LMM(dim)
        self.attention = Attention(dim)

        # Only add output projection if vocab_size given
        if vocab_size:
            self.output_proj = nn.Linear(dim, vocab_size)
        else:
            self.output_proj = None

    def forward(self, tokens):
        memory_vecs = torch.stack([self.lmm(tok) for tok in tokens])
        combined = torch.cat([tokens, memory_vecs], dim=0)
        output, weights = self.attention(combined)

        hidden = output[:len(tokens)]

        # Only project if final layer
        if self.output_proj:
            logits = self.output_proj(hidden)
            return logits, weights
        else:
            return hidden, weights


class DeepMAC(nn.Module):
    def __init__(self, num_layers=3, dim=512, vocab_size=50257):
        super().__init__()

        # Intermediate layers (no vocab projection)
        self.mac_layers = nn.ModuleList([
            MAC_Layer(dim, vocab_size=None) for _ in range(num_layers-1)
        ])

        # Final layer (with vocab projection)
        self.final_mac = MAC_Layer(dim, vocab_size=vocab_size)

    def forward(self, tokens):
        x = tokens

        # Process through intermediate MACs with residual connections
        for mac in self.mac_layers:
            hidden, _ = mac(x)
            x = x + hidden  # Residual: add input to output

        # Final projection to vocab
        logits, weights = self.final_mac(x)

        return logits, weights

In [ ]:
# LMM = Your brain remembering facts
# Attention = Your focus/concentration
# MAC = Using memory + focus together to understand

In [ ]:
def prepare_data(dataset, max_len=50):
    sequences = []

    for i in range(min(100, len(dataset))):  # First 100 stories
        story = dataset[i]

        # Get text
        text = story['text']

        # Tokenize
        tokens = tokenizer.encode(text, add_special_tokens=True)

        # Split into chunks of max_len
        for j in range(0, len(tokens) - max_len, max_len):
            chunk = tokens[j:j + max_len]
            if len(chunk) == max_len:  # Only full chunks
                sequences.append(chunk)

    return sequences

sequences = prepare_data(dataset)
print(f"Prepared {len(sequences)} sequences")
print(f"First sequence length: {len(sequences[0])}")

Prepared 324 sequences
First sequence length: 50


In [ ]:
# Move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
embed = embed.to(device)  # Add this line!

# Move models to GPU
mac = DeepMAC(num_layers=3, dim=512, vocab_size=tokenizer.vocab_size).to(device)
optimizer = torch.optim.Adam(mac.parameters(), lr=0.0003)

criterion = nn.CrossEntropyLoss()

print("\n training Memory Augumented Context (MAC) on Tiny Stories (GPU)...\n")

for epoch in range(50):
    total_loss = 0

    for token_ids in sequences:
        # Move data to GPU
        seq_vecs = embed(torch.tensor(token_ids).to(device))

        # Forward
        logits, _ = mac(seq_vecs)

        # Calculate loss
        loss = 0
        for i in range(len(token_ids) - 1):
            pred_logits = logits[i]
            target_id = torch.tensor([token_ids[i+1]]).to(device)

            loss += criterion(pred_logits.unsqueeze(0), target_id)

        loss = loss / (len(token_ids) - 1)

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mac.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    # Print EVERY epoch
    print(f"Epoch {epoch}: Loss = {total_loss:.2f}")

print("\n Training complete!")

Using device: cuda

 training Memory Augumented Context (MAC) on Tiny Stories (GPU)...

Epoch 0: Loss = 2083.69
Epoch 1: Loss = 1616.76
Epoch 2: Loss = 1495.70
Epoch 3: Loss = 1392.98
Epoch 4: Loss = 1306.10
Epoch 5: Loss = 1219.68
Epoch 6: Loss = 1129.48
Epoch 7: Loss = 1053.94
Epoch 8: Loss = 993.67
Epoch 9: Loss = 940.60
Epoch 10: Loss = 893.45
Epoch 11: Loss = 862.68
Epoch 12: Loss = 816.81
Epoch 13: Loss = 788.20
Epoch 14: Loss = 765.41
Epoch 15: Loss = 723.81
Epoch 16: Loss = 704.33
Epoch 17: Loss = 691.54
Epoch 18: Loss = 668.71
Epoch 19: Loss = 658.93
Epoch 20: Loss = 636.29
Epoch 21: Loss = 626.10
Epoch 22: Loss = 605.30
Epoch 23: Loss = 610.35
Epoch 24: Loss = 581.64
Epoch 25: Loss = 559.39
Epoch 26: Loss = 571.80
Epoch 27: Loss = 548.71
Epoch 28: Loss = 556.69
Epoch 29: Loss = 524.59
Epoch 30: Loss = 518.98
Epoch 31: Loss = 509.76
Epoch 32: Loss = 509.13
Epoch 33: Loss = 496.37
Epoch 34: Loss = 501.07
Epoch 35: Loss = 481.89
Epoch 36: Loss = 474.71
Epoch 37: Loss = 470.84
Ep

In [ ]:
print("\n testing MAC Generation:\n")

def generate_next_word(prompt):
    # Tokenize prompt
    token_ids = tokenizer.encode(prompt)
    seq_vecs = embed(torch.tensor(token_ids).to(device))

    # Generate
    with torch.no_grad():
        logits, _ = mac(seq_vecs)
        next_logits = logits[-1]  # Last position

        # Get most likely token
        next_token_id = torch.argmax(next_logits).item()
        next_word = tokenizer.decode([next_token_id])

    return next_word

# Test prompts
test_prompts = [
    "Once upon a time",
    "The little girl",
    "One day there was",
    "A cat and a dog",
    "The boy wanted to"
]

for prompt in test_prompts:
    next_word = generate_next_word(prompt)
    print(f"'{prompt}' → '{next_word}'")


 testing MAC Generation:

'Once upon a time' → ' there'
'The little girl' → ' alert'
'One day there was' → '.'
'A cat and a dog' → ''s'
'The boy wanted to' → ' away'


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
